In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)




In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [ ]:
# 4. Print shape of one batch
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Batch Image Shape: {images.shape}")
print(f"Batch Label Shape: {labels.shape}")



In [ ]:
# 5. Display sample images
plt.figure(figsize=(10, 5))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    img = images[i].permute(1, 2, 0).numpy()

    plt.imshow(img.astype('uint8'))
    plt.title(f"Age: {int(labels[i].item())}")
    plt.axis('off')
plt.show()



In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
import torch.nn.functional as F

class AgeRegressionModel(nn.Module):
    def __init__(self, input_shape):
        super(AgeRegressionModel, self).__init__()
        self.flatten = nn.Flatten()


        self.fc1 = nn.Linear(input_shape, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 1) # Output: 1 continuous value

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
# Task 2: Write your training loop here:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
# Task 3: Write your validation loop here:
def validate_epoch(model, loader, criterion, device):
    """
    Evaluates the model on the test/validation set.
    """
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

    avg_loss = running_loss / len(loader.dataset)
    return avg_loss



In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

channels, height, width = X_train_tensor.shape[1:]
input_dim = channels * height * width

model = AgeRegressionModel(input_dim).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []

for epoch in range(20):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate_epoch(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/20: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

def plot_performance(train_losses, val_losses):
    plt.figure(figsize=(10, 6))

    plt.plot(train_losses, label='Training Loss', color='#1f77b4', linewidth=2, marker='o', markersize=4)
    plt.plot(val_losses, label='Validation Loss', color='#ff7f0e', linewidth=2, marker='s', markersize=4)

    plt.title('Model Loss Progression (MSE)', fontsize=16, pad=15)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig('loss_plot.png')
    plt.show()

plot_performance(train_losses, val_losses)

In [ ]:
# Task 2 (Bonus): Write your code here:
def plot_sample_predictions(model, loader, device, num_samples=5):
    model.eval()

    images, labels = next(iter(loader))

    with torch.no_grad():
        outputs = model(images.to(device))
        preds = outputs.cpu().numpy()

    plt.figure(figsize=(15, 6))

    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)

        img_tensor = images[i]


        img_plt = img_tensor.permute(1, 2, 0).numpy()


        if img_plt.max() <= 1.01:
            img_plt = img_plt.clip(0, 1)
        else:
            img_plt = img_plt.astype('uint8')

        plt.imshow(img_plt)

        actual = labels[i].item()
        predicted = preds[i][0]
        plt.title(f"Actual: {actual:.0f}\nPred: {predicted:.1f}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

plot_sample_predictions(model, test_loader, device)

